im assuming that l2 is making the problem more tractable for hdbscan, thoda naseeb type. a happy coincidence
I would first like to find all the points with low distance from our already defined cluster on no normalisation. what kind of curve does it form?  

We will use minimum sum for each (single linkage of sorts).    



1. Get all the points which are clustered (patches/pws)
2. for all patches in cat category, find the distance with the closest point in the cluster (single linkage).
3. This is crude, ideally you would want to add the closest point, then the next after refreshing the distance pairs, but its fine for now i think.
4. sort and plot these distances


Ese simple hai, lets start


In [ ]:
%config InteractiveShell.cache_size = 0
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

import lucent
import matplotlib.pyplot as plt
from lucent.optvis import render, param, transform, objectives
from lucent.modelzoo import inceptionv1
from pathlib import Path
import torch
from lucent.optvis.objectives import wrap_objective, handle_batch
from torch.nn import functional as F
import numpy as np
from olt.tfms import transform, inverse_transform
from PIL import Image
from olt.act import InputOutputModelSnapshot
import torch
from olt.html_report import apply_cmap, to_pil, rd_bk_gn
from olt.shards import read_image_shard
import warnings
from tqdm import tqdm
from olt.feature_viz import tensor_to_img_array
from olt.feature_viz import get_feature_viz_input
import pandas as pd
from olt.tfms import transform, inverse_transform
from olt.act import InputOutputModelSnapshot, get_layer_activations
from olt.html_report import make_overlay_heatmap


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


layer_name = "mixed4e_1x1_pre_relu_conv"
channel = 55

plt.style.use("dark_background")

In [ ]:
FLAT_BASE = Path("this-and-prev/flat-images")

report_csv = Path("this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/55/report.csv")
df = pd.read_csv(report_csv)
df = df[df.cluster_label == 61]

df.head()

In [ ]:
patches = []
for tup in tqdm(df.itertuples()):
    batch = transform(Image.open(FLAT_BASE / f"{tup.input_image_key}.jpeg"))[None]
    act = InputOutputModelSnapshot.get_activations(batch, model, [tup.layer_name])[tup.layer_name]["input"]
    patch = act[0, :, tup.y_position, tup.x_position]
    patches.append(patch)    

In [ ]:
w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].reshape(-1)

In [ ]:
from olt.show import show_single_channel_red_green_black as S
S([pws[0].reshape(22,24), pws[1].reshape(22,24)])
plt.show()

In [ ]:
# lets first test on all points in tabby cat, which werent classified
# we would like to not keep the already classified points.  

In [ ]:
layer_name = "mixed4e_1x1_pre_relu_conv"
ikeys = df.input_image_key.unique()
all_patches = []

for ik in tqdm(ikeys):
    clustered_df = df[(df.input_image_key == ik) & (df.cluster_label != -1)]
    clustered_points = set([(t.y_position, t.x_position) for t in clustered_df.itertuples()])
    batch = transform(Image.open(FLAT_BASE / f"{ik}.jpeg"))[None]
    act = InputOutputModelSnapshot.get_activations(batch, model, [layer_name])[layer_name]["input"]    
    for r in range(act.shape[-2]):
        for c in range(act.shape[-1]):
            if (r,c) not in clustered_points:
                all_patches.append(act[0, :, r, c])

In [ ]:
len(all_patches)

In [ ]:
all_pws.shape

In [ ]:
all_pws = np.stack([(p*w).detach().cpu().numpy() for p in all_patches])
pws = np.stack([(p*w).detach().cpu().numpy() for p in patches])

pws.shape, all_pws.shape

In [ ]:
from sklearn.metrics import pairwise_distances

In [ ]:
cosine_mat = pairwise_distances(all_pws, pws, metric="cosine")
euclid_mat = pairwise_distances(all_pws, pws)

In [ ]:
cosine_mat.shape

In [ ]:
points = np.argwhere(min_dists < 0.01)

In [ ]:
# in the end, it doesn't seem very hard to say that something is this or that then
# depending on the ranges, we would have a propositional logic thing only
# the only remaining part is, what do i do with this information lol?
# this is in the end, a huge proposition chain, which maybe i can sort with the distance, but still, a chain
# we assume there would be a set number of chains, this would be the first thing we should test.
# now im again back to ranges btw. 
# for each point in neuron, we basically bin it into a category (or an OR of category). A manual analysis after that would be useful
# this is quite interesting actually
# i basically in the end have a propositional logic of sorts after that.  
# this is a slightly more automatic way to define what a neuron is "saying" at one output (we simply use the ranges yes).

In [ ]:
S([all_pws[points[0]].reshape(22,24), pws[0].reshape(22,24)], 10, viztype="local")

In [ ]:
min_dists = cosine_mat.min(axis=1)
sorteds = np.sort(min_dists)
plt.plot(sorteds[:2000])

In [ ]:
min_dists = euclid_mat.min(axis=1)
sorteds = np.sort(min_dists)
plt.plot(sorteds)